
---

# 🔀 ***`Stacking and Blending Ensembles in Machine Learning for Data Science`***  

---

## ✅ **Introduction to Stacking and Blending**  
📌 **Stacking** and **Blending** are **ensemble learning techniques** that combine the predictions of multiple models to improve performance. Unlike **Bagging** (e.g., Random Forest) and **Boosting** (e.g., AdaBoost, XGBoost), these techniques focus on building **meta-models** that leverage the strengths of multiple base models.  

✅ Useful when individual models have **complementary strengths**.  
✅ Often achieves better performance than standalone models.  
✅ Effective for complex problems with **non-linear relationships**.  

---

## 📊 **1. What is Stacking (Stacked Generalization)?**  

📌 **Stacking** is a two-layer ensemble method that trains a **meta-model** (also called a **blender**) to combine the predictions of multiple **base models**.  

---

### 🔎 **How Does Stacking Work?**  
1️⃣ **Train Base Models:** Train several different models (e.g., Logistic Regression, Random Forest, XGBoost) on the training data.  
2️⃣ **Generate Meta Features:** Each base model makes predictions, and these predictions become the **input features** for the meta-model.  
3️⃣ **Train the Meta-Model:** The meta-model learns to combine the predictions from the base models to make the final prediction.  

---

### 🔄 **Workflow of Stacking**  
```
      ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
      │ Model 1      │    │ Model 2      │    │ Model 3      │
      └─────────────┘    └─────────────┘    └─────────────┘
           ↓                    ↓                    ↓
        Predictions         Predictions           Predictions
           ↓                    ↓                    ↓
           └──────────────→ [ Meta-Model ]  ←──────────────┘
                                ↓
                           Final Prediction
```

---

### 📋 **Python Code: Implementing Stacking in Python**  

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Base Models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Meta-Model (Blender)
meta_model = LogisticRegression()

# Stacking Classifier
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)
stacking_model.fit(X_train, y_train)

# Predictions
y_pred = stacking_model.predict(X_test)

# Model Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9590643274853801

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.94      0.94        63
           1       0.96      0.97      0.97       108

    accuracy                           0.96       171
   macro avg       0.96      0.95      0.96       171
weighted avg       0.96      0.96      0.96       171




---

### 🚀 **Advantages of Stacking**  
✅ Utilizes the strengths of **diverse models**.  
✅ Often achieves **higher accuracy** than individual models.  
✅ Suitable for both **classification** and **regression**.  

### ⚠️ **Limitations of Stacking**  
❌ Training multiple models can be **computationally expensive**.  
❌ Requires careful tuning of **base models** and **meta-model**.  
❌ Risk of **overfitting** if not carefully managed.  

---

## 🔄 **2. What is Blending?**  

📌 **Blending** is another ensemble technique that works similarly to stacking but follows a **simpler strategy**.  

---

### 🔎 **How Does Blending Work?**  
1️⃣ **Train Base Models:** Train multiple models on the training data.  
2️⃣ **Holdout Set Creation:** Split the training data into two parts:  
- **Training Set:** For training base models.  
- **Validation Set:** For generating predictions to train the meta-model.  
  
3️⃣ **Generate Meta Features:** Each base model makes predictions on the **validation set**.  
4️⃣ **Train the Meta-Model:** The meta-model combines the base model predictions to generate the final prediction.  

---

### 🔄 **Workflow of Blending**  
```
           ┌───────────────┐    ┌───────────────┐    ┌───────────────┐
           │ Model 1       │    │ Model 2       │    │ Model 3       │
           └───────────────┘    └───────────────┘    └───────────────┘
                ↓                     ↓                     ↓
         Predictions            Predictions            Predictions
                ↓                     ↓                     ↓
                └───────────→ [ Meta-Model ]  ←──────────────┘
                                      ↓
                                Final Prediction
```

---

### 📋 **Python Code: Implementing Blending in Python**  

In [2]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create Holdout Set (20% for Blending)
X_train_blend, X_valid, y_train_blend, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Base Models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Train Base Models
rf.fit(X_train_blend, y_train_blend)
gb.fit(X_train_blend, y_train_blend)

# Create Meta-Features for Meta-Model
meta_features = np.column_stack([
    rf.predict(X_valid),
    gb.predict(X_valid)
])

# Train Meta-Model
meta_model = LogisticRegression()
meta_model.fit(meta_features, y_valid)

# Predict on Test Data
final_features = np.column_stack([
    rf.predict(X_test),
    gb.predict(X_test)
])

# Final Prediction
y_pred = meta_model.predict(final_features)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9590643274853801




---

### 🚀 **Advantages of Blending**  
✅ Easier to implement than stacking.  
✅ More resistant to **data leakage** since meta-model uses a **holdout set**.  
✅ Requires fewer computational resources compared to stacking.  

### ⚠️ **Limitations of Blending**  
❌ The performance of blending heavily depends on the **holdout set** size.  
❌ Requires careful selection of **base models** to ensure diversity.  

---

## 🔎 **3. Stacking vs. Blending — Key Differences**  

| Feature             | **Stacking** | **Blending** |
|---------------------|---------------|---------------|
| **Training Strategy** | Uses **K-Fold Cross-Validation** to generate meta-features. | Uses a **holdout set** to train the meta-model. |
| **Meta-Model Data** | Meta-model is trained on predictions made by base models on **validation folds**. | Meta-model is trained on predictions made on the **holdout set**. |
| **Complexity** | More complex but often delivers **higher accuracy**. | Simpler and faster to implement. |
| **Risk of Data Leakage** | Less prone to data leakage. | More prone to data leakage if holdout data is too small. |


## 🔥 **4. Best Practices for Stacking and Blending**  

✅ Use **diverse base models** (e.g., Logistic Regression, Random Forest, XGBoost) to maximize performance.  
✅ Select a **simple meta-model** like **Logistic Regression** or **Ridge Regression** to reduce overfitting.  
✅ Ensure **base models are not highly correlated** to maintain diversity.  
✅ For blending, use a **20% holdout set** as meta-data to prevent data leakage.  
✅ Tune hyperparameters using **GridSearchCV** or **RandomizedSearchCV**.  

---

## 📈 **5. Real-World Applications of Stacking & Blending**  

✅ **Finance** → Predicting loan defaults, credit scoring.  
✅ **Healthcare** → Diagnosing diseases based on multiple clinical models.  
✅ **Marketing** → Customer segmentation and churn prediction.  
✅ **NLP (Natural Language Processing)** → Sentiment analysis and text classification.  
✅ **Kaggle Competitions** → Commonly used to win competitions with complex datasets.  

---

## 🏆 **6. Key Takeaways**  

✔️ **Stacking** and **Blending** combine the strengths of multiple models to improve accuracy.  
✔️ **Stacking** excels in complex data with multiple feature interactions.  
✔️ **Blending** is simpler and faster but may be less powerful than stacking.  
✔️ Use **cross-validation** to fine-tune models and avoid overfitting.  

💡 **Pro Tip:** Start with simpler ensemble techniques (like Bagging or Boosting) before experimenting with Stacking and Blending for complex datasets.

---


### **Further Stacking Practice**

In [1]:
# import libaries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score

In [2]:
# load the data

df = df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day68-stacking-and-blending/heart.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [3]:
X = df.drop(columns=['target'])
y = df['target']

In [5]:
X.sample(5)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
110,64,0,0,180,325,0,1,154,1,0.0,2,0,2
13,64,1,3,110,211,0,0,144,1,1.8,1,0,2
165,67,1,0,160,286,0,0,108,1,1.5,1,3,2
20,59,1,0,135,234,0,1,161,0,0.5,1,0,3
16,58,0,2,120,340,0,1,172,0,0.0,2,0,2


In [6]:
y.sample(5)

94     1
282    0
212    0
33     1
91     1
Name: target, dtype: int64

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=8)

In [8]:
print(X_train.shape)
print(y_train.shape)

(242, 13)
(242,)


In [9]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=10)),
    ('gbdt',GradientBoostingClassifier())
]

In [10]:
clf = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression(),
    cv=10
)

In [11]:
clf.fit(X_train, y_train)

StackingClassifier(cv=10,
                   estimators=[('rf',
                                RandomForestClassifier(n_estimators=10,
                                                       random_state=42)),
                               ('knn', KNeighborsClassifier(n_neighbors=10)),
                               ('gbdt', GradientBoostingClassifier())],
                   final_estimator=LogisticRegression())

In [12]:
y_pred = clf.predict(X_test)

In [13]:
accuracy_score(y_test,y_pred)

0.8688524590163934

---